# シナリオ1 デモ用データセット & 想定QA集（サポート対応時間の長期化要因分析）

Genie Agent のボリュームファイル分析デモ「シナリオ1：対応時間が長期化している要因の分析」用に、複数テーブル（CSV）× 複数ボリュームファイル（PDF/DOCX/PPTX/JPG）で、様々な角度から分析できるデータセットと想定QAをまとめました。

## 1. データセット構成

### テーブルデータ（CSV・5ファイル）

複数テーブルに分割し、Genie Agent がJOINを跨いで多角的に分析できるようにしています。

| ファイル | 役割 | 主なカラム | 件数 |
|---|---|---|---|
| support_tickets.csv | ファクト（チケット） | ticket_id, customer_id, product_id, agent_id, category, priority, channel, open_date, close_date, resolution_time_hours, reopened_count, csat_score, status | 60件 |
| customers.csv | ディメンション（顧客） | customer_id, customer_name, segment, region, industry, plan_tier, mrr_jpy, contract_start_date | 30社 |
| agents.csv | ディメンション（担当者） | agent_id, agent_name, team, tier, location, hire_date | 10名 |
| products.csv | ディメンション（製品） | product_id, product_name, module, release_version | 5製品 |
| sla_policies.csv | リファレンス（SLA基準） | priority, sla_target_hours, business_hours_only | 4行 |

カテゴリは6種類（データ同期エラー／ログイン認証エラー／決済・請求エラー／パフォーマンス低下／UI表示崩れ／機能に関する質問）。「データ同期エラー」だけが平均解決時間・再発率・SLA違反率・CSATのいずれも著しく悪化するように設計しており、テーブルだけでは「何が悪いか」は分かるが「なぜ」は分からない状態になっています。

### ボリュームファイル（非構造化・4ファイル、前回作成分と同一）

いずれも「データ同期エラー」の中心ケース（TICK-10204 ほか、顧客 さくらリテール株式会社／CUST-0007）に関する文書で、ここに根本原因（外部連携APIプロバイダ Connector社による無通知のレート制限変更）が記述されています。

- `escalation_email_TICK-10204.pdf`：顧客からのエスカレーションメール
- `tier2_handoff_notes_TICK-10204.docx`：Tier1→Tier2 引継ぎ調査メモ
- `data_sync_incident_review.pptx`：社内インシデントレビュー資料（RCA）
- `error_screenshot_TICK-10204.jpg`：顧客環境のエラー画面スクリーンショット

## 2. 想定QA（分析軸ごと）

各軸について、**Step1：テーブルのみで分かること**と、**Step2：ボリューム追加で分かること**を整理しています。軸によっては「テーブルだけで十分に答えられる」ものと、「テーブルだけでは本当の理由にたどり着けない」ものを両方含め、Genie Agent のボリューム機能の価値がどこで発揮されるかが分かるようにしています。

### 軸1：カテゴリ別の解決時間

* **Step1（テーブルのみ）**：「カテゴリ別の平均解決時間を教えて」
  → 回答例：データ同期エラーが平均61.6時間で最長。次いでパフォーマンス低下12.8時間、決済/請求エラー14.1時間、ログイン認証エラー9.7時間、UI表示崩れ5.2時間、機能に関する質問3.1時間。
* **Step2（ボリューム追加）**：「データ同期エラーの解決時間が他カテゴリより突出して長い理由は？」
  → PDF/DOCXの記述から、外部連携先 Connector社が事前通知なくAPIレート制限を変更したため、CloudBridge側の同期バッチが対応できず再発を繰り返していたことが根本原因と分かる。

### 軸2：SLA遵守状況（優先度別）

* **Step1（テーブルのみ）**：「優先度ごとのSLA違反率が最も高いカテゴリはどこ？」
  → 回答例：データ同期エラーはSLA違反率100%（10件中10件がSLA目標を超過）。パフォーマンス低下50%、決済/請求エラー40%、ログイン認証エラー20%、UI表示崩れ10%、機能に関する質問0%。
* **Step2（ボリューム追加）**：「データ同期エラーでSLAを守れなかった具体的な経緯は？」
  → DOCXの調査タイムラインから、Tier1での通常対応では解決できず、Tier2・エンジニアリングへのエスカレーションと外部要因の調査に時間を要したことがSLA超過の直接的な経緯として分かる。

### 軸3：チーム／担当者別の対応状況

* **Step1（テーブルのみ）**：「チームごとの平均解決時間と対応件数は？」
  → 回答例：Tier1 Supportは平均9.0時間（50件）、Tier2 Supportは平均63.6時間（5件）、Engineeringは平均59.6時間（5件）。Tier2・Engineeringが対応する件数は少ないが、1件あたりの対応時間が非常に長い。
* **Step2（ボリューム追加）**：「Tier2・Engineeringへのエスカレーションが必要になった技術的な理由は？」
  → DOCXから、Tier1の標準手順（再実行）では解決せず、外部API仕様変更の調査という高度な技術対応が必要だったことが分かる。

### 軸4：顧客満足度（CSAT）との関係

* **Step1（テーブルのみ）**：「カテゴリ別の平均CSATスコアを教えて」
  → 回答例：データ同期エラーは平均1.4（5点満点）で最も低い。他カテゴリは4.1〜4.7と高水準。
* **Step2（ボリューム追加）**：「CSATが低いデータ同期エラーのチケットで、顧客は具体的に何に不満を述べているか？」
  → PDFのエスカレーションメール内の顧客コメント（「同じ問題が3回も繰り返されており…契約更新の判断を保留せざるを得ない」）から、再発そのものと恒久対応の欠如への不満が低CSATの直接要因であることが分かる。

### 軸5：顧客セグメント・業種・地域別の傾向

* **Step1（テーブルのみ）**：「セグメント別の平均解決時間は？」
  → 回答例：Enterpriseセグメントが平均28.9時間と最長（Mid-Market 11.9時間、SMB 15.5時間と比べて突出）。
* **Step2（ボリューム追加）**：「Enterpriseセグメントの解決時間が長くなっている背景は？」
  → テーブル側で「データ同期エラー10件中5件がさくらリテール株式会社（エンタープライズ・CUST-0007）に集中している」ことまでは分かるが、なぜさくらリテール株式会社でこれほど集中しているかはPDF/DOCXのConnector社レート制限変更という根本原因まで見ないと説明できない。

### 軸6：プロダクト／モジュール別の傾向

* **Step1（テーブルのみ）**：「製品モジュール別の平均解決時間は？」
  → 回答例：CloudBridge Sync Engineが平均51.9時間で最長。他モジュール（Auth Gateway 8.6時間、Analytics 10.7時間、Billing 12.9時間、Workspace UI 4.8時間）と比べて大幅に悪い。
* **Step2（ボリューム追加）**：「Sync Engineの障害が集中している根本原因は？」
  → PPTXのRCAスライドから、外部APIのレート制限変更に対しSync Engine側のリトライロジックが追従していなかったという製品側の技術的要因が分かる。

### 軸7：チャネル別の解決時間比較

* **Step1（テーブルのみ、ボリューム不要で十分なケース）**：「問い合わせチャネル別の平均解決時間は？」
  → 回答例：Email経由が平均21.8時間と最長、Web Form 19.5時間、Chat 16.0時間、Phone 12.2時間と最短。この傾向はテーブルデータのみで十分に説明可能（Emailは複雑な事案のエスカレーションに使われやすいなど）。

### 軸8：再発（reopened_count）が多いチケットの特徴

* **Step1（テーブルのみ）**：「再発回数が2回以上のチケットにはどんな共通点があるか？」
  → 回答例：該当する5件のうち4件が「データ同期エラー」カテゴリで、そのうち3件がさくらリテール株式会社（CUST-0007）。
* **Step2（ボリューム追加）**：「これらの再発チケットがすべて同じ根本原因なのか？」
  → DOCX/PPTXの記述から、いずれもConnector社のAPIレート制限変更に起因する同一の構造的問題であり、個別の対応ではなく恒久対応（exponential backoff実装）が必要であることが分かる。

### 軸9：特定顧客ディープダイブ（さくらリテール株式会社／CUST-0007）

* **Step1（テーブルのみ）**：「さくらリテール株式会社が起票したチケットの傾向は？」
  → 回答例：7件中5件が「データ同期エラー」でUrgent/High優先度、平均解決時間は60時間超、CSATは1〜2と低水準。決済/請求エラーは2件のみで解決時間・CSATともに良好。
* **Step2（ボリューム追加）**：「さくらリテール株式会社がエスカレーションで要求している内容と、対応状況は？」
  → PDFから「同じ問題の3回目の再発」「月次レポート業務への支障」「契約更新判断の保留」という要求・懸念が、PPTXから対応策（exponential backoff実装、担当：Data Sync Platformチーム、期限：10/15）と進捗が分かる。

### 軸10：時系列トレンド

* **Step1（テーブルのみ）**：「さくらリテール株式会社のデータ同期エラーは時間の経過でどう変化しているか？」
  → 回答例：8/18 → 9/4 → 9/13 → 9/18 → 9/20 と、9月に入ってから再発の間隔が狭まっており、短期間での再発頻度が加速している。
* **Step2（ボリューム追加）**：「なぜ9月以降に再発頻度が加速したのか？」
  → DOCXから、9月8日付でConnector社がAPIレート制限を変更したことが直接の起点であり、それ以降に同期失敗が繰り返し発生するようになったことが分かる。

## 3. まとめ

カテゴリ・SLA・CSAT・セグメント・プロダクト・再発傾向・時系列など、ほぼすべての量的分析軸が「データ同期エラー」および「さくらリテール株式会社」に収束するように設計しています。テーブルのみでは一貫して「何が悪いか（What）」までしか分からず、ボリューム追加後に初めて「なぜ悪いのか（Why）」＝Connector社のAPIレート制限変更という共通の根本原因に到達できる、という一貫したストーリーでデモを進行できます。
